In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from typing import Tuple, Optional, Dict, List
import requests
import zipfile
import os
import io
from pathlib import Path
import cv2
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import h5py

In [ ]:
# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import h5py
import h5py.h5r # Explicitly import h5r
import numpy as np
import gdown
import os

class GoogleDriveNYUDepthDataset(Dataset):
    """NYUDepthV2 dataset loader for Google Drive dataset"""

    def __init__(self, mode='train', img_size=(224, 224),
                 drive_link=None, data_dir='./nyu_data'):
        """
        Args:
            mode: 'train' or 'val'
            img_size: Tuple for image size (height, width)
            drive_link: Google Drive link for dataset
            data_dir: Directory to store downloaded data
        """
        self.mode = mode
        self.img_size = img_size
        self.drive_link = drive_link or "https://drive.google.com/file/d/1GdYTZkjeT3391XOw-7rfJrFaXQU1zFom/view?usp=drive_link"
        self.data_dir = data_dir

        # Image transforms
        self.img_transform = transforms.Compose([
            transforms.Resize(img_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])

        # Depth transforms - use nearest neighbor
        self.depth_transform = transforms.Compose([
            transforms.Resize(img_size, interpolation=Image.NEAREST),
            transforms.ToTensor()
        ])

        # Load dataset
        self._load_dataset()

    def _download_dataset(self):
        """Download dataset from Google Drive"""
        os.makedirs(self.data_dir, exist_ok=True)

        print(f"Downloading dataset from Google Drive...")

        # Extract file ID from Google Drive link
        file_id = self.drive_link.split('/d/')[1].split('/')[0]

        # Define output path
        output_path = os.path.join(self.data_dir, 'nyu_depth_v2_labeled.mat')

        # Download using gdown
        url = f'https://drive.google.com/uc?id={file_id}'
        gdown.download(url, output_path, quiet=False)

        print(f"Dataset downloaded to: {output_path}")
        return output_path

    def _load_dataset(self):
        """Load dataset from Google Drive or local cache"""

        # Check if file already exists
        mat_path = os.path.join(self.data_dir, 'nyu_depth_v2_labeled.mat')

        if not os.path.exists(mat_path):
            mat_path = self._download_dataset()

        print(f"Loading dataset from {mat_path}...")

        # Open the .mat file and keep it open
        self.f = h5py.File(mat_path, 'r')

        # Get references to images and depths
        # These are now references to datasets, not the data itself
        self.images_ref = self.f['images']  # Shape: [N]
        self.depths_ref = self.f['depths']  # Shape: [N]

        # Get dataset dimensions (number of samples)
        total_samples = self.images_ref.shape[0] # Number of references

        # Split dataset: 80% train, 20% validation
        if self.mode == 'train':
            self.start_idx = 0
            self.end_idx = int(total_samples * 0.8)
        else:  # 'val'
            self.start_idx = int(total_samples * 0.8)
            self.end_idx = total_samples

        self.length = self.end_idx - self.start_idx

        print(f"Loaded {self.length} samples for {self.mode} mode")

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # Calculate actual index in the full dataset
        actual_idx = self.start_idx + idx

        # --- Handle Image Data ---
        img_data_candidate = self.images_ref[actual_idx]
        if isinstance(img_data_candidate, h5py.h5r.Reference):
            # It's a direct HDF5 reference
            img_data = self.f[img_data_candidate][()]
        elif isinstance(img_data_candidate, np.ndarray):
            # It's a numpy array. It could contain a reference or be the actual image data.
            try:
                potential_ref = img_data_candidate.item()
                if isinstance(potential_ref, h5py.h5r.Reference):
                    # It was a numpy array wrapping a single reference
                    img_data = self.f[potential_ref][()]
                else:
                    # It was a numpy array, but its item was not a reference, so it's the data itself
                    img_data = img_data_candidate
            except ValueError:
                # .item() failed, meaning it's a multi-element numpy array, so it's the data itself
                img_data = img_data_candidate
            except Exception as e:
                raise RuntimeError(f"Error processing image data candidate (ndarray) at index {actual_idx}: {e}") from e
        else:
            # Fallback for unexpected types, assuming it's a direct reference if not an array.
            raise TypeError(f"Unexpected type for image data at index {actual_idx}: {type(img_data_candidate)}")

        # Transpose to [H, W, 3] and convert to uint8
        img_np = np.transpose(img_data, (1, 2, 0)).astype(np.uint8)

        # --- Handle Depth Data ---
        depth_data_candidate = self.depths_ref[actual_idx]
        if isinstance(depth_data_candidate, h5py.h5r.Reference):
            depth_np = self.f[depth_data_candidate][()].astype(np.float32)
        elif isinstance(depth_data_candidate, np.ndarray):
            try:
                potential_ref = depth_data_candidate.item()
                if isinstance(potential_ref, h5py.h5r.Reference):
                    depth_np = self.f[potential_ref][()].astype(np.float32)
                else:
                    depth_np = depth_data_candidate.astype(np.float32)
            except ValueError:
                depth_np = depth_data_candidate.astype(np.float32)
            except Exception as e:
                raise RuntimeError(f"Error processing depth data candidate (ndarray) at index {actual_idx}: {e}") from e
        else:
            raise TypeError(f"Unexpected type for depth data at index {actual_idx}: {type(depth_data_candidate)}")


        # Normalize depth to [0, 1] range
        depth_np = np.clip(depth_np, 0, 10) / 10.0

        # Convert to PIL Images
        img_pil = Image.fromarray(img_np)
        depth_pil = Image.fromarray((depth_np * 255).astype(np.uint8)) # Convert to uint8 for PIL

        # Apply transforms
        img_tensor = self.img_transform(img_pil)
        # Removed depth_tensor.squeeze(0) to keep channel dimension for consistency with model output
        depth_tensor = self.depth_transform(depth_pil)

        return img_tensor, depth_tensor

In [ ]:
class ViTOcclusionDetector(nn.Module):
    """ViT-based feature extractor with occlusion detection"""

    def __init__(self, patch_size=16, embed_dim=256):
        super().__init__()

        self.patch_size = patch_size
        self.embed_dim = embed_dim

        # Patch embedding
        self.patch_embed = nn.Conv2d(3, embed_dim, kernel_size=patch_size,
                                     stride=patch_size)

        # Position embedding
        num_patches = (224 // patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim) * 0.02)

        # Transformer encoder 
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=8,
            dim_feedforward=1024,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=6)

        # Occlusion detection head
        self.occlusion_head = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

        # CLS token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

    def forward(self, x):
        """x: [B, 3, 224, 224]"""
        B = x.shape[0]

        # Patch embedding
        x = self.patch_embed(x)  # [B, D, H/p, W/p]
        x = x.flatten(2).transpose(1, 2)  # [B, N, D]

        # Add CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        # Add position embedding
        x = x + self.pos_embed

        # Transformer
        x = self.transformer(x)

        # Separate CLS and patch tokens
        cls_token = x[:, 0]  # CLS token for global features
        patch_tokens = x[:, 1:]  # Patch tokens

        # Predict occlusion mask per patch
        occlusion_mask = self.occlusion_head(patch_tokens).squeeze(-1)  # [B, N]

        return patch_tokens, occlusion_mask, cls_token

In [ ]:
class OcclusionReasoner(nn.Module):
    """Reason about occluded regions using transformer attention"""

    def __init__(self, feature_dim=256, hidden_dim=128):
        super().__init__()

        self.visible_encoder = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim)
        )

        self.context_encoder = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim)
        )

        # Cross-attention between visible and occluded regions
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=4,
            dropout=0.1,
            batch_first=True
        )

        self.output_proj = nn.Linear(hidden_dim, feature_dim)

    def forward(self, patch_features, occlusion_mask):
        """
        Args:
            patch_features: [B, N, D]
            occlusion_mask: [B, N] (1 = occluded, 0 = visible)
        """
        B, N, D = patch_features.shape

        # Separate visible and occluded features
        visible_mask = (occlusion_mask < 0.5).unsqueeze(-1)  # [B, N, 1]
        occluded_mask = (occlusion_mask >= 0.5).unsqueeze(-1)

        visible_features = patch_features * visible_mask
        occluded_features = torch.zeros_like(patch_features)

        # Encode features
        visible_encoded = self.visible_encoder(visible_features)
        context_encoded = self.context_encoder(patch_features)

        # Use visible features as query, all features as context
        hallucinated, _ = self.cross_attention(
            query=visible_encoded,
            key=context_encoded,
            value=context_encoded
        )

        # Combine with original occluded features
        hallucinated = self.output_proj(hallucinated)

        # Replace occluded regions with hallucinated features using torch.where
        output_features = torch.where(occluded_mask.expand_as(patch_features), hallucinated, patch_features)

        return output_features

In [ ]:
class TimeEmbedding(nn.Module):
    """Sinusoidal time embedding"""

    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        emb = torch.log(torch.tensor(10000.0)) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t.float().unsqueeze(1) * emb.unsqueeze(0)
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
        return emb

class DiffusionDenoiser(nn.Module):
    """Diffusion denoising network"""

    def __init__(self, in_channels=1, feat_dim=256, time_dim=128):
        super().__init__()

        self.time_embed = TimeEmbedding(time_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64)
        )

        self.feat_mlp = nn.Sequential(
            nn.Linear(feat_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64)
        )

        # CNN for denoising
        self.conv1 = nn.Conv2d(in_channels + 128, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv3 = nn.Conv2d(128, 64, 3, padding=1)
        self.conv4 = nn.Conv2d(64, in_channels, 3, padding=1)

        self.norm1 = nn.BatchNorm2d(64)
        self.norm2 = nn.BatchNorm2d(128)
        self.norm3 = nn.BatchNorm2d(64)

    def forward(self, x, features, t, occlusion_mask=None):
        """
        x: [B, 1, H, W] - noisy depth
        features: [B, D] - image features
        t: [B] - timestep
        """
        B, _, H, W = x.shape

        # Time embedding
        t_emb = self.time_embed(t)  # [B, time_dim]
        t_emb = self.time_mlp(t_emb)  # [B, 64]
        t_emb = t_emb.view(B, 64, 1, 1).expand(-1, -1, H, W)

        # Feature embedding
        f_emb = self.feat_mlp(features)  # [B, 64]
        f_emb = f_emb.view(B, 64, 1, 1).expand(-1, -1, H, W)

        # Concatenate all inputs
        x_in = torch.cat([x, t_emb, f_emb], dim=1)

        # Denoising CNN
        x1 = F.relu(self.norm1(self.conv1(x_in)))
        x2 = F.relu(self.norm2(self.conv2(x1)))
        x3 = F.relu(self.norm3(self.conv3(x2)))
        x_out = torch.sigmoid(self.conv4(x3))

        return x_out

In [ ]:
class OcclusionAware3DRecon(nn.Module):
    """Complete pipeline optimized for Colab"""

    def __init__(self, img_size=(224, 224)):
        super().__init__()

        self.img_size = img_size

        # 1. Feature extraction with occlusion detection
        self.feature_extractor = ViTOcclusionDetector()

        # 2. Occlusion reasoning
        self.occlusion_reasoner = OcclusionReasoner()

        # 3. Diffusion model
        self.diffusion = DiffusionDenoiser()

        # 4. Depth decoder
        self.depth_decoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            # nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), # Removed this line
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 1, 1),
            nn.Sigmoid()
        )

        # 5. Feature aggregator
        self.feature_aggregator = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 256)
        )

    def forward(self, x, t=None, mode='train'):
        """
        Args:
            x: Input image [B, 3, H, W]
            t: Timestep for diffusion [B]
            mode: 'train' or 'inference'
        """
        B = x.shape[0]

        # Step 1: Extract features and occlusion mask
        patch_features, occlusion_mask, global_feature = self.feature_extractor(x)

        # Step 2: Occlusion reasoning
        reasoned_features = self.occlusion_reasoner(patch_features, occlusion_mask)

        # Aggregate features
        aggregated_feature = reasoned_features.mean(dim=1)
        aggregated_feature = self.feature_aggregator(aggregated_feature)

        if mode == 'train' and t is not None:
            # Training: Add noise and denoise
            # Generate initial partial depth (from visible regions)
            with torch.no_grad():
                visible_mask = (occlusion_mask < 0.5).float().view(B, 1, 14, 14)
                visible_mask = F.interpolate(visible_mask, size=self.img_size,
                                           mode='nearest')

                # Simple depth initialization (closer objects are brighter)
                partial_depth = visible_mask * 0.5

            # Add noise
            noise = torch.randn_like(partial_depth)
            noisy_depth = partial_depth + (t.float()[:, None, None, None] / 1000) * noise

            # Denoise
            denoised = self.diffusion(noisy_depth, aggregated_feature, t)

        else:
            # Inference: Start from random noise and denoise iteratively
            noisy_depth = torch.randn(B, 1, *self.img_size, device=x.device) * 0.5 + 0.5

            if mode == 'inference':
                # Iterative denoising (simplified - 5 steps for speed)
                for i in range(5):
                    t_step = torch.full((B,), 100 - i*20, device=x.device)
                    noisy_depth = self.diffusion(noisy_depth, aggregated_feature, t_step)
            else:
                # Single step for validation
                t_step = torch.zeros(B, device=x.device)
                noisy_depth = self.diffusion(noisy_depth, aggregated_feature, t_step)

            denoised = noisy_depth

        # Final depth refinement
        complete_depth = self.depth_decoder(denoised)

        return {
            'depth': complete_depth,
            'occlusion_mask': occlusion_mask,
            'partial_depth': denoised,
            'features': aggregated_feature
        }

In [ ]:
def create_sample_batch(batch_size=4):
    """Create a sample batch for testing"""
    images = torch.randn(batch_size, 3, 224, 224)
    depths = torch.rand(batch_size, 1, 224, 224)
    return images, depths

class ColabTrainer:
    """Training class optimized for Colab"""

    def __init__(self, model, device='cuda'):
        self.model = model.to(device)
        self.device = device
        self.optimizer = optim.AdamW(model.parameters(), lr=1e-4)
        self.scheduler = optim.lr_scheduler.StepLR(self.optimizer, step_size=10, gamma=0.5)

        # Loss functions
        self.depth_loss = nn.L1Loss()
        self.mse_loss = nn.MSELoss()

        # Tracking
        self.losses = []

    def train_step(self, images, depths):
        """Single training step"""
        self.model.train()
        self.optimizer.zero_grad()

        images = images.to(self.device)
        depths = depths.to(self.device)

        # Random timestep
        B = images.shape[0]
        t = torch.randint(0, 100, (B,), device=self.device)

        # Forward pass
        outputs = self.model(images, t, mode='train')

        # Compute loss
        loss = self.depth_loss(outputs['depth'], depths)

        # Backward
        loss.backward()
        self.optimizer.step()

        return loss.item()

    def validate(self, images, depths):
        """Validation step"""
        self.model.eval()
        with torch.no_grad():
            images = images.to(self.device)
            depths = depths.to(self.device)

            outputs = self.model(images, mode='val')
            loss = self.depth_loss(outputs['depth'], depths)

        return loss.item(), outputs

In [ ]:
def visualize_results(images, depths_gt, depths_pred, occlusion_mask, num_samples=2):
    """Visualize model predictions"""
    num_samples = min(num_samples, images.shape[0])

    fig, axes = plt.subplots(num_samples, 5, figsize=(20, num_samples*4))

    if num_samples == 1:
        axes = axes.reshape(1, -1)

    for i in range(num_samples):
        # Input image
        img_np = images[i].permute(1, 2, 0).cpu().numpy()
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())
        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title('Input Image')
        axes[i, 0].axis('off')

        # Ground truth depth
        depth_gt_np = depths_gt[i, 0].cpu().numpy()
        axes[i, 1].imshow(depth_gt_np, cmap='plasma')
        axes[i, 1].set_title('GT Depth')
        axes[i, 1].axis('off')

        # Predicted depth
        depth_pred_np = depths_pred[i, 0].cpu().numpy()
        axes[i, 2].imshow(depth_pred_np, cmap='plasma')
        axes[i, 2].set_title('Pred Depth')
        axes[i, 2].axis('off')

        # Occlusion mask
        mask_np = occlusion_mask[i].cpu().numpy().reshape(14, 14)
        axes[i, 3].imshow(mask_np, cmap='hot')
        axes[i, 3].set_title('Occlusion Mask')
        axes[i, 3].axis('off')

        # Error map
        error = np.abs(depth_gt_np - depth_pred_np)
        axes[i, 4].imshow(error, cmap='Reds')
        axes[i, 4].set_title('Error Map')
        axes[i, 4].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
def run_complete_pipeline():
    
    print("=" * 70)
    print("OCCLUSION-AWARE GENERATIVE FRAMEWORK FOR 3D RECONSTRUCTION")
    print("=" * 70)

    # 1. Initialize everything
    print("\n1. Initializing model and data...")
    model = OcclusionAware3DRecon().to(device)

    # Create datasets using REAL NYUDepthV2 data
    print("Loading real NYUDepthV2 dataset...")
    train_dataset = GoogleDriveNYUDepthDataset(mode='train')
    val_dataset = GoogleDriveNYUDepthDataset(mode='val')

    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

    # 2. Initialize trainer
    print("2. Initializing trainer...")
    trainer = ColabTrainer(model, device)

    # 3. Quick test of forward pass
    print("3. Testing forward pass...")
    # Get samples from the real dataset
    test_images, test_depths = next(iter(train_loader))

    test_images = test_images.to(device)

    with torch.no_grad():
        outputs = model(test_images, mode='inference')
    print(f"   Model forward pass successful")
    print(f"   Output depth shape: {outputs['depth'].shape}")
    print(f"   Occlusion mask shape: {outputs['occlusion_mask'].shape}")

    # 4. Training loop (short version for Colab)
    print("\n4. Starting training (5 epochs for demo)....")
    num_epochs = 5
    for epoch in range(num_epochs):
        # Training
        epoch_loss = 0
        for batch_idx, (images, depths) in enumerate(train_loader):
            loss = trainer.train_step(images, depths)
            epoch_loss += loss

            if batch_idx % 10 == 0:
                print(f"   Epoch {epoch+1}, Batch {batch_idx}: Loss = {loss:.4f}")

        avg_loss = epoch_loss / len(train_loader)
        print(f"   Epoch {epoch+1} Average Loss: {avg_loss:.4f}")

        # Validation
        val_images, val_depths = next(iter(val_loader))
        val_loss, val_outputs = trainer.validate(val_images, val_depths)
        print(f"   Validation Loss: {val_loss:.4f}")

        # Update scheduler
        trainer.scheduler.step()

    # 5. Visualization
    print("\n5. Visualizing results...")
    sample_images, sample_depths = next(iter(val_loader))
    sample_images = sample_images.to(device)

    with torch.no_grad():
        sample_outputs = model(sample_images, mode='inference')

    visualize_results(
        sample_images.cpu(),
        sample_depths.cpu(),
        sample_outputs['depth'].cpu(),
        sample_outputs['occlusion_mask'].cpu(),
        num_samples=4
    )

    # 6. Save model
    print("\n6. Saving model...")
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': trainer.optimizer.state_dict(),
    }, 'occlusion_3d_model_colab.pth')
    print(" Model saved as 'occlusion_3d_model_colab.pth'")


    return model, trainer

In [ ]:
if __name__ == "__main__":
    # Run the pipeline
    model, trainer = run_complete_pipeline()
    # Additional: Test inference on single image
    print("\n" + "=" * 70)
    print("SINGLE IMAGE INFERENCE DEMO:")
    print("=" * 70)

    # Create a test image
    test_image = torch.randn(1, 3, 224, 224).to(device)

    with torch.no_grad():
        model.eval()
        result = model(test_image, mode='inference')

        depth_map = result['depth'][0, 0].cpu().numpy()
        occlusion_mask = result['occlusion_mask'][0].cpu().numpy()

        print(f"Depth map range: [{depth_map.min():.3f}, {depth_map.max():.3f}]")
        print(f"Occlusion rate: {(occlusion_mask > 0.5).mean():.2%}")
        print(" Inference successful!")